# 01 — Exploratory Data Analysis

First pass at the ULB credit card fraud dataset. We're after three things in this notebook:

1. **Class balance** — quantify how severe the imbalance is.
2. **Feature distributions** — `Amount` and `Time` are raw; `V1`..`V28` are PCA-anonymized.
3. **Temporal patterns** — does fraud rate vary across the 48-hour window the data covers?

Follow-up correlations and fraud-vs-normal feature comparisons land in notebook 02.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from fraud_shield.config import settings

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 40)
RANDOM_SEED = settings.random_seed

Loading the raw CSV. Run `make data` first if you hit the assertion below.

In [ ]:
csv_path = settings.data_raw / "creditcard.csv"
assert csv_path.exists(), f"Run `make data` first — {csv_path} not found"

df = pd.read_csv(csv_path)
df.shape

In [ ]:
df.head()

In [ ]:
df.info(memory_usage="deep")

## Class balance

Fraud is the minority class. Quantifying how minority is the first thing to look at — it dictates which metrics, sampling strategies, and threshold-tuning approaches we should use later.

In [ ]:
counts = df["Class"].value_counts()
pcts = df["Class"].value_counts(normalize=True) * 100
pd.DataFrame({"count": counts, "pct": pcts.round(3)})

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x="Class", ax=ax, palette=["#4c72b0", "#dd8452"])
ax.set_yscale("log")
ax.set_title("Class distribution (log scale)")
for patch, count in zip(ax.patches, counts.values):
    ax.annotate(
        f"{count:,}",
        (patch.get_x() + patch.get_width() / 2, patch.get_height()),
        ha="center", va="bottom",
    )
plt.tight_layout()
plt.show()

0.17% fraud rate (492 / 284,807). At this skew, accuracy is meaningless — a constant `predict 0` gets 99.83% accuracy and zero recall. Plan: report PR-AUC and recall at a fixed precision, not accuracy or ROC-AUC.

## Feature distributions

Time and Amount are raw fields; V1..V28 are the principal components from the ULB team's PCA — already centered, so no scaling story to tell there. The interesting raw feature is Amount, which is heavily right-skewed and benefits from a log transform.

In [ ]:
df[["Time", "Amount"]].describe().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["Amount"], bins=50, ax=axes[0], color="#4c72b0")
axes[0].set_yscale("log")
axes[0].set_title("Amount (raw, log-y)")

sns.histplot(np.log1p(df["Amount"]), bins=50, ax=axes[1], color="#4c72b0")
axes[1].set_title("log1p(Amount)")
plt.tight_layout()
plt.show()

In [ ]:
sample_v = ["V1", "V4", "V10", "V14", "V17"]
fig, axes = plt.subplots(1, len(sample_v), figsize=(15, 3))
for ax, col in zip(axes, sample_v):
    sns.kdeplot(data=df, x=col, hue="Class", common_norm=False, ax=ax, fill=True, alpha=0.4)
    ax.set_title(col)
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

## Temporal patterns

The `Time` column is seconds elapsed since the first transaction — the dataset spans roughly 48 hours. Converting to hour-of-day lets us check whether fraud rate is uniform across the daily cycle or clusters at specific times.

In [ ]:
df["hour"] = ((df["Time"] // 3600) % 24).astype(int)

hourly = df.groupby("hour").agg(
    n=("Class", "size"),
    n_fraud=("Class", "sum"),
)
hourly["fraud_rate_pct"] = (hourly["n_fraud"] / hourly["n"] * 100).round(3)
hourly

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

sns.barplot(x=hourly.index, y=hourly["n"], ax=ax1, color="#4c72b0")
ax1.set_title("Transactions per hour-of-day")
ax1.set_xlabel("hour")
ax1.set_ylabel("transactions")

sns.lineplot(x=hourly.index, y=hourly["fraud_rate_pct"], ax=ax2, marker="o", color="#dd8452")
ax2.set_title("Fraud rate (%) per hour-of-day")
ax2.set_xlabel("hour")
ax2.set_ylabel("% fraud")

plt.tight_layout()
plt.show()

## Initial observations

*(Fill in after running the cells above against the real dataset.)*

- Class balance: ~0.17% fraud. Below 1% — deep imbalance, ROC-AUC will mislead.
- `Amount` is heavily right-skewed; `log1p(Amount)` is much more model-friendly.
- The PCA features `V1..V28` are centered around zero. Several (e.g. V14, V17) show visibly different distributions between classes — strong candidate features.
- Volume drops sharply during nighttime hours; fraud rate appears to be **higher** at low-volume hours — a useful signal.

## Next

Notebook 02 (Day 3 of the roadmap) picks up:

- Correlations between V1..V28 (and any that correlate with `Amount`).
- Side-by-side fraud-vs-normal stats per feature, with Cohen's d to rank separability.
- The pandera schema that will guard the data pipeline.